# FER2013: Facial Expression Recognition

სახის ემოციების კლასიფიკაცია PyTorch-ის გამოყენებით. 3 არქიტექტურა, 2 hyperparameter sweep, wandb ლოგირება.

## 1. გარემოს მომზადება

სანამ რამეს გავუშვებ, ჯერ ყველაფერს ერთ ადგილას უნდა მოვუყარო თავი: კოდი GitHub-იდან ჩამოვიწერო, საჭირო ბიბლიოთეკები დავაყენო, შევიყვანო Kaggle-ისა და wandb-ის API KEY და ბოლოს თავად მონაცემებიც ჩამოვტვირთო. ეს ნაწილი მთელ მოსამზადებელ სამუშაოს აკეთებს.

In [ ]:
import os, getpass

# რეპოზიტორია
!git clone https://github.com/GigiSichinava/ML-Assignment-4.git /content/ML-Assignment-4 2>/dev/null || echo 'already cloned'
os.chdir('/content/ML-Assignment-4')
!pip install -q wandb kaggle

# credentials
os.environ['KAGGLE_USERNAME'] = input('kaggle username: ').strip()
os.environ['KAGGLE_KEY']      = getpass.getpass('kaggle key: ').strip()
os.environ['WANDB_API_KEY']   = getpass.getpass('wandb key: ').strip()

# მონაცემები
import pandas as pd
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    !kaggle datasets download -d deadskull7/fer2013 -p data/
    !unzip -q -o data/fer2013.zip -d data/
    df = pd.read_csv('data/fer2013.csv')
    df[df['Usage']=='Training'][['emotion','pixels']].to_csv('data/train.csv', index=False)
    df[df['Usage']!='Training'][['pixels']].to_csv('data/test.csv', index=False)
    print('train.csv და test.csv შეიქმნა')
else:
    print('მონაცემები უკვე ადგილზეა')

# დაკავშირება wandb-თან
import wandb
wandb.login()

# იმპორტები
import torch
from src.config import Config, PRESETS
from src.data import get_dataloaders
from src.models import get_model
from src.train import train_model, make_submission
from src.utils import set_seed, count_params, check_initial_loss, overfit_small_batch, plot_history, plot_confusion

device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(42)
print('ready, device:', device)

## 2. Sanity Check-ები (Forward და Backward)

სანამ რეალურ ვარჯიშზე გადავალ, მინდა დავრწმუნდე, რომ მოდელი და training loop გამართულია. ჯერ ვამოწმებ, საწყისი loss იმ მნიშვნელობასთან ახლოს არის თუ არა, რასაც თეორიულად ველოდები (random მოდელზე ~ ln(7) = 1.946), შემდეგ კი ვცდილობ მოდელმა სულ რამდენიმე მაგალითი ზეპირად დაიმახსოვროს. თუ ორივე გამოდის, ე.ი. forward და backward pass სწორად მუშაობს.


In [ ]:
cfg = Config(arch='SmallCNN')
train_loader, val_loader, test_loader, class_weights = get_dataloaders(cfg)
model = get_model(cfg).to(device)
print('params:', count_params(model))

# forward შემოწმება
check_initial_loss(model, train_loader, device)

# backward შემოწმება
overfit_small_batch(get_model(cfg).to(device), train_loader, device, n=20, steps=200)

## 3. ექსპერიმენტი 1: BaselineMLP

ვიწყებ ყველაზე მარტივი მოდელით: fully-connected ქსელით, რომელიც სურათს ერთ გრძელ ვექტორად შლის. წინასწარვე ვიცი, რომ ის სუსტი იქნება, რადგან flatten-ის შემდეგ იკარგება spatial ინფორმაცია, ანუ რომელი პიქსელი რომელ პიქსელს ეკვრის. ეს განზრახ მინდა baseline-ად, რომელსაც შემდეგ მოდელებს შევადარებ.


In [ ]:
cfg = PRESETS['mlp_baseline']
model_mlp, history, best, (y_true, y_pred) = train_model(cfg, device=device)
plot_history(history, title='BaselineMLP')
print('best val acc:', round(best, 3))

## 4. ექსპერიმენტი 2: SmallCNN

ახლა გადავდივარ convolutional ქსელზე, რომელიც, MLP-სგან განსხვავებით, სურათის სტრუქტურას ინარჩუნებს და მის სხვადასხვა ნაწილში ლოკალურ ნიმუშებს ეძებს. ორი conv block-ისა და BatchNorm-ის შემდეგ ველოდები, რომ მარტივ მოდელთან შედარებით val accuracy შესამჩნევად აიწევს.

In [ ]:
cfg = PRESETS['smallcnn']
model_small, history, best, (y_true, y_pred) = train_model(cfg, device=device)
plot_history(history, title='SmallCNN')
plot_confusion(y_true, y_pred)
print('best val acc:', round(best, 3))

## 5. ექსპერიმენტი 3: DeeperCNN (Regularization-ის გარეშე)

აქ ქსელს ვამძიმებ: ოთხ conv block-ს ვაწყობ, ოღონდ განზრახ არ ვრთავ არც dropout-ს და არც augmentation-ს. მინდა ვაჩვენო, რა ხდება, როცა ძლიერ მოდელს თავისუფლად ვუშვებ: train accuracy თითქმის 100%-ს უახლოვდება, val კი მკვეთრად ჩამორჩება და val loss იწყებს ზრდას. სწორედ ეს არის overfitting-ის სუფთა მაგალითი.


In [ ]:
cfg = PRESETS['deepcnn_overfit']
model_deep_ov, history, best, _ = train_model(cfg, device=device)
plot_history(history, title='DeeperCNN (no regularization)')
print('best val acc:', round(best, 3))

## 6. ექსპერიმენტი 4: DeeperCNN (Regularized)

ახლა იმავე ღრმა ქსელს ვუმატებ რეგულარიზაციას: dropout-ს, weight decay-ს, augmentation-სა და cosine learning rate scheduler-ს. მიზანი ისაა, რომ წინა ცდის overfitting შევამცირო: overfit_gap დავწიო და val accuracy ავწიო ისე, რომ მოდელი მაინც კარგად განაზოგადებდეს.


In [ ]:
cfg = PRESETS['deepcnn_regularized']
model_best, history, best, (y_true, y_pred) = train_model(cfg, device=device)
plot_history(history, title='DeeperCNN (regularized)')
plot_confusion(y_true, y_pred)
print('best val acc:', round(best, 3))

## 7. Hyperparameter Sweep: SmallCNN, სხვადასხვა Learning Rate

მხოლოდ არქიტექტურის შერჩევა საკმარისი არ არის, დიდი მნიშვნელობა აქვს იმასაც, რა ტემპით სწავლობს მოდელი. აქ SmallCNN-ს სამი სხვადასხვა learning rate-ით ვუშვებ, რომ დავინახო, რომელია ზედმეტად მაღალი (არასტაბილური), რომელი ზედმეტად დაბალი (ნელი კონვერგენცია) და სად არის ოქროს შუალედი.


In [ ]:
for lr in [1e-2, 1e-3, 1e-4]:
    cfg = Config(arch='SmallCNN', run_name=f'smallcnn_lr{lr}',
                 lr=lr, dropout=0.25, epochs=25)
    _, history, best, _ = train_model(cfg, device=device)
    print(f'lr={lr}: best val acc = {round(best, 3)}')

## 8. Hyperparameter Sweep: DeeperCNN, სხვადასხვა Dropout

ამჯერად ღრმა ქსელზე dropout-ის სხვადასხვა მნიშვნელობას ვამოწმებ. მაინტერესებს, როგორ მოქმედებს ის train-სა და val accuracy-ს შორის სხვაობაზე, ანუ პირდაპირ overfit_gap-ზე. ველოდები, რომ dropout=0.0 მაღალ accuracy-ს მოგვცემს დიდი gap-ით, dropout=0.5 კი პატარა gap-ით, ოღონდ ოდნავ დაბალი accuracy-თ.


In [ ]:
for d in [0.0, 0.3, 0.5]:
    cfg = Config(arch='DeeperCNN', run_name=f'deepcnn_drop{d}',
                 dropout=d, weight_decay=1e-4, augment=True,
                 scheduler='cosine', epochs=40)
    _, history, best, _ = train_model(cfg, device=device)
    print(f'dropout={d}: best val acc = {round(best, 3)}')

## 9. Kaggle Submission

ბოლოს საუკეთესო მოდელით (regularized DeeperCNN) test set-ზე პროგნოზს ვაკეთებ და შედეგს იმ ფორმატში ვინახავ, რომელსაც Kaggle ელოდება.

In [ ]:
make_submission(model_best, PRESETS['deepcnn_regularized'],
                out_path='submission.csv', device=device)